# Finance Transaction Risk Analytics — EDA
Exploratory Data Analysis on 50,000+ banking transactions and customer data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Data

In [ ]:
# Update paths as needed
txn  = pd.read_csv('finance_transactions.csv')
cust = pd.read_csv('customers.csv')

print('Transactions shape:', txn.shape)
print('Customers shape   :', cust.shape)

## 2. Data Cleaning

In [ ]:
# Strip column names
txn.columns  = txn.columns.str.strip()
cust.columns = cust.columns.str.strip()
cust.rename(columns={'fisrt_name': 'first_name'}, inplace=True)

# Strip string whitespace
for col in txn.select_dtypes('object'):  txn[col]  = txn[col].str.strip()
for col in cust.select_dtypes('object'): cust[col] = cust[col].str.strip()

# Parse dates
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'], dayfirst=True, errors='coerce')
cust['date_of_birth']   = pd.to_datetime(cust['date_of_birth'],   dayfirst=True, errors='coerce')
cust['join_date']       = pd.to_datetime(cust['join_date'],       dayfirst=True, errors='coerce')

# Fix numeric nulls
txn['fee_amount'] = pd.to_numeric(txn['fee_amount'], errors='coerce').fillna(0)
txn['tax_amount'] = pd.to_numeric(txn['tax_amount'], errors='coerce').fillna(0)
txn['risk_score'] = pd.to_numeric(txn['risk_score'], errors='coerce').fillna(txn['risk_score'].median())

# Binary flags
txn['fraud_flag']  = txn['is_fraud'].str.lower().map({'yes': 1, 'no': 0}).fillna(0).astype(int)
txn['failed_flag'] = (txn['transaction_status'].str.lower() == 'failed').astype(int)

# Merge
df = txn.merge(cust, on='customer_id', how='left')

print(f'Merged shape      : {df.shape}')
print(f'Null values total : {df.isnull().sum().sum()}')

In [ ]:
df.head(5)

In [ ]:
df.dtypes

In [ ]:
df.describe(include='all')

## 3. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

plt.figure(figsize=(10, 4))
missing.plot(kind='bar', color='steelblue')
plt.title('Missing Values per Column')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
ref = df['transaction_date'].max()

df['txn_year']      = df['transaction_date'].dt.year
df['txn_month']     = df['transaction_date'].dt.month
df['txn_dow']       = df['transaction_date'].dt.dayofweek
df['txn_quarter']   = df['transaction_date'].dt.quarter
df['age']           = ((ref - df['date_of_birth']).dt.days / 365.25).round(1)
df['tenure_months'] = ((ref - df['join_date']).dt.days / 30.44).round(1)
df['total_cost']    = df['amount'] + df['fee_amount'] + df['tax_amount']
df['fee_rate']      = (df['fee_amount'] / df['amount'].replace(0, np.nan)).fillna(0).round(4)
df['high_amount']   = (df['amount'] > df['amount'].quantile(0.90)).astype(int)

cust_agg = df.groupby('customer_id').agg(
    cust_txn_count  = ('transaction_id', 'count'),
    cust_avg_amount = ('amount', 'mean'),
    cust_fail_rate  = ('failed_flag', 'mean'),
    cust_fraud_rate = ('fraud_flag', 'mean'),
).reset_index()
df = df.merge(cust_agg, on='customer_id', how='left')

print('New features added: txn_year, txn_month, txn_dow, txn_quarter, age, tenure_months,')
print('                    total_cost, fee_rate, high_amount, cust_txn_count, cust_avg_amount,')
print('                    cust_fail_rate, cust_fraud_rate')

## 5. Univariate Analysis

In [ ]:
# Transaction amount distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['amount'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount (INR)')

axes[1].hist(df['amount'].clip(upper=df['amount'].quantile(0.99)), bins=60, color='teal', edgecolor='white')
axes[1].set_title('Amount Distribution (99th pct clip)')
axes[1].set_xlabel('Amount (INR)')
plt.tight_layout()
plt.show()

In [ ]:
# Risk score distribution
plt.figure(figsize=(8, 4))
sns.histplot(df['risk_score'], bins=40, kde=True, color='coral')
plt.title('Risk Score Distribution')
plt.xlabel('Risk Score')
plt.show()

In [ ]:
# Categorical value counts
cat_cols = ['transaction_type', 'transaction_status', 'channel', 'merchant_category', 'customer_segment']
fig, axes = plt.subplots(1, len(cat_cols), figsize=(20, 4))
for ax, col in zip(axes, cat_cols):
    vc = df[col].value_counts().head(10)
    vc.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(col)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. Fraud & Failure Rates

In [ ]:
print(f"Fraud Rate  : {df['fraud_flag'].mean()*100:.2f}%")
print(f"Failure Rate: {df['failed_flag'].mean()*100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df['fraud_flag'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%',
    labels=['No Fraud', 'Fraud'], colors=['#3b82d4', '#ef4444'])
axes[0].set_title('Fraud Split')
axes[0].set_ylabel('')

df['failed_flag'].value_counts().plot.pie(ax=axes[1], autopct='%1.1f%%',
    labels=['Success', 'Failed'], colors=['#22c55e', '#ef4444'])
axes[1].set_title('Failure Split')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 7. Monthly Transaction Trend

In [ ]:
monthly = df.groupby(['txn_year', 'txn_month'])['amount'].sum().reset_index()
monthly['period'] = monthly['txn_year'].astype(str) + '-' + monthly['txn_month'].astype(str).str.zfill(2)

plt.figure(figsize=(14, 4))
plt.plot(monthly['period'], monthly['amount'], marker='o', color='steelblue')
plt.xticks(rotation=45)
plt.title('Monthly Transaction Volume (INR)')
plt.xlabel('Period')
plt.ylabel('Total Amount')
plt.tight_layout()
plt.show()

## 8. Customer Behaviour Analysis

In [ ]:
# By segment
seg = df.groupby('customer_segment').agg(
    txn_count  = ('transaction_id', 'count'),
    avg_amount = ('amount', 'mean'),
    fraud_rate = ('fraud_flag', 'mean'),
    fail_rate  = ('failed_flag', 'mean'),
).round(3)
print('--- By Segment ---')
print(seg)

seg['txn_count'].plot(kind='bar', color='steelblue', figsize=(8, 4), title='Transactions per Segment')
plt.tight_layout()
plt.show()

In [ ]:
# By channel
ch = df.groupby('channel').agg(
    txn_count = ('transaction_id', 'count'),
    fail_rate = ('failed_flag', 'mean'),
).sort_values('txn_count', ascending=False).round(3)
print('--- By Channel ---')
print(ch)

In [ ]:
# Fraud rate by merchant category
mc = df.groupby('merchant_category')['fraud_flag'].mean().sort_values(ascending=False)
mc.plot(kind='bar', color='tomato', figsize=(12, 4), title='Fraud Rate by Merchant Category')
plt.ylabel('Fraud Rate')
plt.tight_layout()
plt.show()

## 9. Correlation Heatmap

In [ ]:
num_cols = ['amount', 'fee_amount', 'tax_amount', 'risk_score',
            'fee_rate', 'total_cost', 'age', 'tenure_months',
            'cust_txn_count', 'cust_avg_amount', 'cust_fail_rate',
            'fraud_flag', 'failed_flag']

plt.figure(figsize=(12, 9))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 10. Outlier Detection (Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['amount', 'fee_amount', 'risk_score']):
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', color='navy'))
    ax.set_title(f'Boxplot: {col}')
    ax.set_ylabel(col)
plt.tight_layout()
plt.show()

## 11. Age & Tenure Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['age'].dropna(), bins=30, color='teal', edgecolor='white')
axes[0].set_title('Customer Age Distribution')
axes[0].set_xlabel('Age (years)')

axes[1].hist(df['tenure_months'].dropna(), bins=30, color='orchid', edgecolor='white')
axes[1].set_title('Customer Tenure Distribution')
axes[1].set_xlabel('Tenure (months)')
plt.tight_layout()
plt.show()

## 12. Day-of-Week & Monthly Patterns

In [ ]:
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_counts = df['txn_dow'].value_counts().sort_index()
dow_counts.index = day_labels

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dow_counts.plot(kind='bar', ax=axes[0], color='steelblue', title='Transactions by Day of Week')
axes[0].set_xlabel('Day')
axes[0].tick_params(axis='x', rotation=0)

month_counts = df['txn_month'].value_counts().sort_index()
month_counts.plot(kind='bar', ax=axes[1], color='coral', title='Transactions by Month')
axes[1].set_xlabel('Month')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## Summary
- Dataset merged from transactions + customers; cleaned dates, numeric nulls, and string whitespace.
- Engineered features: age, tenure, total_cost, fee_rate, high_amount, txn_month, txn_dow, customer-level aggregates.
- Key observations documented above — proceed to `model.py` for ML pipeline.